# Credit Risk Scorecard

## Setup and Preprocessing

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# --- Machine learning ---
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score,
    confusion_matrix, fbeta_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from imblearn.metrics import geometric_mean_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline

# --- UMAP (optional; falls back to t-SNE) ---
try:
    import umap
    UMAP_AVAILABLE = True
    print("UMAP available — will use UMAP for Diagnostic 3c.")
except Exception:
    UMAP_AVAILABLE = False
    print("UMAP not available — will use t-SNE for Diagnostic 3c.")
from sklearn.manifold import TSNE

# --- Plot style ---
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12

RANDOM_STATE = 42
MAJORITY = "Good"
MINORITY = "Default"

print("Libraries loaded successfully.")

**Note:** Download the dataset from Kaggle: https://www.kaggle.com/competitions/home-credit-default-risk/data?select=application_train.csv and place it in a `data/` folder.

In [ ]:
# load and check the dataset
df = pd.read_csv("data/application_train.csv")
df.head()

### Handling missing values

In [ ]:
# Checking missing values
# Display all rows without truncating
pd.set_option('display.max_rows', None)

# Shows the percentage of missing values per column
missing_pct = (df.isna().sum() / len(df)) * 100
# Show every column, but only if it contains at least one missing value
print(missing_pct.loc[missing_pct > 0])

# Reset the option back to default after
pd.reset_option('display.max_rows')

In [ ]:
# Drops columns with majority of its data missing
# Identify columns with more than 50% missing data
cols_to_drop = missing_pct[missing_pct > 50].index.tolist()

# Drop those columns all at once
df_cleaned = df.drop(columns=cols_to_drop)

# Verify how many columns were removed
print(f"Dropped {len(cols_to_drop)} columns.")
print(f"Remaining columns: {df_cleaned.shape[1]}")

Dropped columns with majority (>50%) of its data missing because they cannot be realistically filled.

In [ ]:
# Check the skewness of every numerical column and apply the appropriate imputation method to preserve dataset size
# Select only numerical columns
numerical_cols = df_cleaned.select_dtypes(include=['number']).columns

for col in numerical_cols:
    # Skip if there are no missing values to save time
    if df_cleaned[col].isna().sum() == 0:
        continue
        
    # Calculate skewness for the column
    col_skew = df_cleaned[col].skew()
    
    if -0.5 <= col_skew <= 0.5:
        # Normally distributed -> Fill with Mean
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mean())
        # print(f"Imputed {col} with MEAN (Skewness: {col_skew:.2f})")
    else:
        # Skewed -> Fill with Median
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())
        # print(f"Imputed {col} with MEDIAN (Skewness: {col_skew:.2f})")

In [ ]:
# For categorical data, replace missing values with a string like 'Unknown'
# Explicitly flagging as missing
df_cleaned['OCCUPATION_TYPE'] = df_cleaned['OCCUPATION_TYPE'].fillna('Unknown')

### Converting categorical variables to dummy variables

In [ ]:
# Drops the first category of each variable automatically to prevent the "dummy variable trap."
df_encoded = pd.get_dummies(df_cleaned, drop_first=True, dtype=int)

### Plots showing income distribution across key variables

In [ ]:
# Take a safe sample of your large dataset to prevent memory lag
df_sample = df_cleaned[['AMT_INCOME_TOTAL', 'CODE_GENDER']].sample(n=100000, random_state=42, replace=True)

plt.figure(figsize=(10, 5))

# Create a violin plot to see the density shape, or a boxplot for quartiles
sns.boxplot(data=df_sample, x='CODE_GENDER', y='AMT_INCOME_TOTAL')

# CRUCIAL FOR INCOME: Use a log scale if you have massive income outliers
plt.yscale('log') 

plt.title('Distribution of Income by Gender')
plt.xlabel('Gender')
plt.ylabel('Total Income (Log Scale)')
plt.show()

#### Plot 1
**Distribution of Income by Gender**
> **Median:** Visually inspecting the median line in each gender, it is clear that the total median income of males are slightly higher than females.

> **Spread/Variance:** Both genders have compact, tight boxes, indicating that there is not that much variation (uniform) in income and that each gender has predicatable earnings.

> **Outliers:** In both genders, there are a significant amount of outliers as can be seen from the individual dots extending far above the box's whiskers. These individual dots can be described as **ultra-high earners**.

> **Unknown Gender Labels:** `XNA` uncleaned categorical variables.

In [ ]:
# 1. Handle outliers by filtering for the 99th percentile
# (Essential for financial data, otherwise extreme values squish the plot)
q_income = df_cleaned['AMT_INCOME_TOTAL'].quantile(0.99)
q_credit = df_cleaned['AMT_CREDIT'].quantile(0.99)
df_filtered = df_cleaned[(df_cleaned['AMT_INCOME_TOTAL'] < q_income) & (df_cleaned['AMT_CREDIT'] < q_credit)]

# 2. Plot the Hexbin
plt.figure(figsize=(10, 6))
hb = plt.hexbin(df_filtered['AMT_INCOME_TOTAL'], 
                df_filtered['AMT_CREDIT'], 
                gridsize=40, 
                cmap='Blues', 
                bins='log') # Log scale for density colors handles uneven distribution

cb = plt.colorbar(hb, label='Log10(Count of Borrowers)')
plt.title('Density Distribution: Income vs. Credit Amount')
plt.xlabel('Total Income (AMT_INCOME_TOTAL)')
plt.ylabel('Credit Amount (AMT_CREDIT)')
plt.show()

#### Plot 2
**Density Distribution: Income vs. Credit Amount**
> **Core Borrowers:** Dark blue hexagons sit around the lower region of both axes, and this indicates the exact income and credit amount the majority of borrowers have.

> **Distribution:** The dark blue hexagons show no uniform shape, and there's a region where they stretch over in one line, meaning that one variable varies widely while the other is fixed.

> **Isolated Dark Islands:** Dark hexagons can be seen sitting around areas of mostly light coloured hexagons. These isolated dark hexagons shows that there is a massive spike of identical values far away from the norm.

In [ ]:
# 1. Take a safe sample and filter out the top 1% outliers across all three
q_annuity = df_cleaned['AMT_ANNUITY'].quantile(0.99)
df_clean = df_cleaned[
    (df_cleaned['AMT_INCOME_TOTAL'] < q_income) & 
    (df_cleaned['AMT_CREDIT'] < q_credit) & 
    (df_cleaned['AMT_ANNUITY'] < q_annuity)
]
df_sample = df_clean.sample(n=50000, random_state=42)

# 2. Plot using transparency (alpha) to manage overlapping dots
plt.figure(figsize=(12, 7))
scatter = plt.scatter(df_sample['AMT_INCOME_TOTAL'], 
                      df_sample['AMT_CREDIT'], 
                      c=df_sample['AMT_ANNUITY'], 
                      cmap='viridis', 
                      alpha=0.4, 
                      s=10) # 's' controls dot size

cb = plt.colorbar(scatter, label='Annuity Amount (AMT_ANNUITY)')
plt.title('Relationship: Income vs. Credit (Colored by Annuity Amount)')
plt.xlabel('Total Income')
plt.ylabel('Credit Amount')
plt.show()

#### Plot 3
**Relationship: Income vs. Credit**
> **Baseline Trend:** The diagonal shows a spread out positive correlation, affirming that borrows with higher income have a higher credit amount.

> **Annuity:** Annuity (the loan payment amount) increases with credit amount, as can be seen in its shift from dark purple around the lower portion of the graph to a yellow-green colour.

> **Ceiling:** There is a clear edge at the topmost part of the scatter plot where datapoints do not cross, indicating that there exists a hard cut-off of the allowed credit amount.

> **Striping:** Along the vertical axis, there are dense lines of dots, indicating that many records have the same value. This is due to the imputation step where numerical columns with majority of its data missing were either filled with their mean or median.

> **Impossible Loans:** Found along the lower end of the plot, a few high annuity (yellow-green) dots can be seen. This indicates that few have a low credit limit but very high annuity.

---

## Imbalance

In [ ]:
# --- Label-encode all remaining categorical columns ---
cat_cols = df_cleaned.select_dtypes(include="object").columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_cleaned[col] = le.fit_transform(df_cleaned[col])

### D1: Imbalance Severity

In [ ]:
y = df_cleaned['TARGET']
X = df_cleaned.drop(columns=['TARGET'])

n_total = len(y)
n_pos   = (y == 1).sum()   # Minority
n_neg   = (y == 0).sum()   # Majority
pct_pos = n_pos / n_total * 100
pct_neg = n_neg / n_total * 100
IR      = n_neg / n_pos

if   IR < 4:   severity = "Mild"
elif IR < 9:   severity = "Moderate"
elif IR < 100: severity = "Severe"
else:          severity = "Extreme"

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([MAJORITY, MINORITY], [n_neg, n_pos],
            color=["#4C72B0", "#C44E52"], alpha=0.85)
for bar, val in zip(axes[0].patches, [n_neg, n_pos]):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + n_total * 0.005,
                 f"{val:,}", ha="center", fontsize=11)
axes[0].set_title("Class Counts", fontweight="bold")
axes[0].set_ylabel("Borrowers")

axes[1].bar([MAJORITY, MINORITY], [pct_neg, pct_pos],
            color=["#4C72B0", "#C44E52"], alpha=0.85)
for bar, val in zip(axes[1].patches, [pct_neg, pct_pos]):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=11)
axes[1].set_title("Class Proportions", fontweight="bold")
axes[1].set_ylabel("Percentage (%)")
axes[1].axhline(50, color="gray", linestyle="--", linewidth=1.2,
                label="Perfect balance (50%)")
axes[1].legend(fontsize=9)

plt.suptitle(f"Diagnostic 1 — Severity: {severity} | IR = {IR:.1f}:1",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# --- Summary block ---
print("=" * 55)
print("DIAGNOSTIC 1 — IMBALANCE SEVERITY")
print("=" * 55)
print(f"  Total records        : {n_total:,}")
print(f"  Majority    : {n_neg:,} ({pct_neg:.2f}%)")
print(f"  Minority   : {n_pos:,} ({pct_pos:.2f}%)")
print(f"  Imbalance Ratio (IR) : {IR:.1f}:1")
print(f"  Severity class       : {severity}")

The Home Credit Default Risk dataset has a 282,686/24,825 split where `8.07%` of borrowers defaulted while `91.93%` did not. With an IR ≈ 11.4:1, imbalance falls in the **Severe** range. Thus, treatment is **warranted**.

### D2: Scarcity vs. Prior Probability Skew

In [ ]:
n_features = X.shape[1]
epv        = n_pos / n_features

if   n_pos < 100:   verdict = "GENUINE SCARCITY"
elif n_pos < 1000:  verdict = "BORDERLINE"
else:               verdict = "PRIOR SKEW DOMINANT"

epv_flag = "OK" if epv >= 10 else "LOW — scarcity concern"

# --- Plot ---
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(["EPV (this dataset)"], [epv],
        color="#55A868" if epv >= 10 else "#C44E52", alpha=0.85)
ax.axvline(10, color="red", linestyle="--", linewidth=2,
           label="EPV = 10 (minimum threshold)")
ax.text(epv + 0.3, 0, f"{epv:.1f}", va="center", fontsize=12, fontweight="bold")
ax.set_xlabel("Minority Records per Feature (EPV)")
ax.set_title("Diagnostic 2 — Events Per Variable Check", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# --- Summary block ---
print("=" * 55)
print("DIAGNOSTIC 2 — SCARCITY vs. PRIOR PROBABILITY SKEW")
print("=" * 55)
print(f"  Minority count   : {n_pos:,} (defaulted borrowers)")
print(f"  Number of feats  : {n_features}")
print(f"  EPV              : {epv:.1f}  [{epv_flag}]")
print(f"  Verdict          : {verdict}")

The count of the minority class is well over 1,000, indicating that it has enough examples but the volume of the majority class dominates training. Thus, there's a need to correct the distribution, either through `undersampling` or `class weighting`.

The Events per Variable (EPV), which describes the minority records per feature, is greater than 10, showing that the minority class meets the minimum for stable model learning.

### D3: Class Overlap and Boundary Complexity

In [ ]:
# --- 3a: KS test per feature ---
X_pos = df_cleaned[df_cleaned['TARGET'] == 1].drop(columns=['TARGET'])
X_neg = df_cleaned[df_cleaned['TARGET'] == 0].drop(columns=['TARGET'])

ks_records = []
for feat in X.columns:
    stat, pval = ks_2samp(X_pos[feat].dropna(), X_neg[feat].dropna())
    ks_records.append({"feature": feat, "KS_stat": stat, "p_value": pval})

ks_df = (pd.DataFrame(ks_records)
         .sort_values("KS_stat", ascending=False)
         .reset_index(drop=True))

high_sep  = (ks_df["KS_stat"] >= 0.30).sum()
mod_sep   = ((ks_df["KS_stat"] >= 0.10) & (ks_df["KS_stat"] < 0.30)).sum()
high_over = (ks_df["KS_stat"] < 0.10).sum()

print(f"  High separation  (KS >= 0.30) : {high_sep} features")
print(f"  Moderate overlap (0.10–0.30)  : {mod_sep} features")
print(f"  High overlap     (KS < 0.10)  : {high_over} features")
print()
print("  Top 5 most separable features:")
print(ks_df.head(5)[["feature", "KS_stat"]].to_string(index=False))

# --- Bar chart ---
colors_ks = ["#C44E52" if v < 0.10 else "#DD8452" if v < 0.30 else "#55A868"
             for v in ks_df["KS_stat"]]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(ks_df["feature"], ks_df["KS_stat"], color=colors_ks, alpha=0.85)
ax.axvline(0.10, color="orange", linestyle="--", linewidth=1.5,
           label="KS=0.10 (high overlap zone)")
ax.axvline(0.30, color="green",  linestyle="--", linewidth=1.5,
           label="KS=0.30 (high separation zone)")
ax.set_xlabel("KS Statistic")
ax.set_title("Diagnostic 3a — Feature-Level Class Overlap (KS Test)\n"
             "Red=high overlap | Orange=moderate | Green=well separated",
             fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

Majority of the features have **high overlap at the boundary**, necessitating Borderline-SMOTE.

In [ ]:
# --- 3b: Density histograms — 3 most separable vs. 3 most overlapping ---
top_sep  = ks_df.head(3)["feature"].tolist()
top_over = ks_df.tail(3)["feature"].tolist()
plot_feats = top_sep + top_over

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(plot_feats):
    ks_val = ks_df.loc[ks_df["feature"] == feat, "KS_stat"].values[0]
    axes[i].hist(X_neg[feat].dropna(), bins=40, alpha=0.55,
                 color="#4C72B0", label="Good", density=True)
    axes[i].hist(X_pos[feat].dropna(), bins=40, alpha=0.55,
                 color="#C44E52", label="Defaulted", density=True)
    axes[i].set_title(f"{feat} | KS = {ks_val:.3f}", fontsize=9, fontweight="bold")
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel("Value", fontsize=8)
    axes[i].set_ylabel("Density", fontsize=8)

plt.suptitle("Diagnostic 3b — Class Distributions\n"
             "Top row: most separable | Bottom row: most overlapping",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

The most separable features are `EXT_SOURCE_2`, `EXT_SOURCE_3`, and `DAYS_BIRTH`, while the most overlapping features are `FLAG_DOCUMENT_20`, `FLAG_DOCUMENT_12`, and `FLAG_MOBIL`.

In [ ]:
# --- OPTIMIZATION 1: Downsample for visualization ---
# 307k points is too dense to see anyway. 10k-20k is plenty.
SAMPLE_SIZE = 15000 
if len(X) > SAMPLE_SIZE:
    # Stratified sampling to keep the exact "Defaulted" ratio intact
    from sklearn.model_selection import train_test_split
    X_sample, _, y_sample, _ = train_test_split(
        X, y, train_size=SAMPLE_SIZE, stratify=y, random_state=RANDOM_STATE
    )
else:
    X_sample, y_sample = X, y

# Process the smaller sample
X_vals = X_sample.fillna(0).values
y_vals = y_sample.values

# Scaler and PCA run instantly on sampled data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_vals)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_ * 100

# UMAP / t-SNE now takes seconds instead of hours
if UMAP_AVAILABLE:
    print("Running UMAP on sample...")
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE,
                        n_neighbors=30, min_dist=0.1, n_jobs=-1) # Added n_jobs=-1
    X_embed = reducer.fit_transform(X_scaled)
    embed_label = "UMAP"
    dim_labels = ("UMAP Dim 1", "UMAP Dim 2")
else:
    print("Running t-SNE on sample...")
    # --- OPTIMIZATION 2: Use multi-core n_jobs=-1 ---
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, 
                max_iter=1000, n_jobs=-1) 
    X_embed = tsne.fit_transform(X_scaled)
    embed_label = "t-SNE"
    dim_labels = ("t-SNE Dim 1", "t-SNE Dim 2")

# --- Side-by-side: PCA and UMAP/t-SNE ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, coords, title, xl, yl in [
    (axes[0], X_pca, "PCA 2D Projection",
     f"PC1 ({var_exp[0]:.1f}% var)", f"PC2 ({var_exp[1]:.1f}% var)"),
    (axes[1], X_embed, f"{embed_label} 2D Projection",
     dim_labels[0], dim_labels[1])
]:
    # --- OPTIMIZATION 3: Plot the majority class (Good) first so Defaulted sits on top ---
    for label, color, name in [(0, "#4C72B0", "Good"), (1, "#C44E52", "Defaulted")]:
        mask = y_vals == label
        
        # Use rasterized=True to prevent matplotlib from lagging on large vector outputs
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=color, alpha=0.4, s=10 if label == 0 else 20,
                   label=f"{name} (n={mask.sum():,})",
                   zorder=2 if label == 1 else 1,
                   rasterized=True) 
        
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(f"Diagnostic 3c — {title}", fontweight="bold")
    ax.legend(fontsize=9, markerscale=2)

plt.tight_layout()
plt.show()

- **KS Statistics:**  
  None of the features achieved “high separation” (KS ≥ 0.30). The strongest signals were `EXT_SOURCE_2` (0.223 and `EXT_SOURCE_3` (0.219), followed by `DAYS_BIRTH` (0.122). These fall into the *moderate overlap* range, meaning they provide some discriminatory power but not enough to cleanly separate defaulters from non‑defaulters. Moreover, `NAME_INCOME_TYPE` (0.106) and `DAYS_LAST_PHONE_CHANGE` (0.097) also showed moderate signal but were not carried forward into the engineered feature set or final pipeline.   

- **2D Projections:**  
  The PCA/UMAP plots show heavy overlap between defaulted and non‑defaulted borrowers. This confirms there is **no clean, linearly separable boundary** in the feature space. The overlap supplements the Severe IR finding.

The absence of highly separable features and the heavy overlap together highlight why **Borderline‑SMOTE** is necessary in breaking the collapse caused by imbalance and overlap, so the model can start recognizing defaults.

### D4: Baseline Classifier Behaviour

In [ ]:
# --- OPTIMIZATION 1: Downsample for visualization ---
SAMPLE_SIZE = 15000 
if len(X) > SAMPLE_SIZE:
    from sklearn.model_selection import train_test_split
    X_sample, _, y_sample, _ = train_test_split(
        X, y, train_size=SAMPLE_SIZE, stratify=y, random_state=RANDOM_STATE
    )
else:
    X_sample, y_sample = X, y

# Process the smaller sample
X_vals = X_sample.fillna(0).values
y_vals = y_sample.values

# Scaler runs on raw values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_vals)

# --- NEW: Apply Borderline-SMOTE to the scaled visualization data ---
# This generates synthetic 'Defaulted' points specifically along the class boundaries
bsmote = BorderlineSMOTE(sampling_strategy='auto', kind='borderline-1', random_state=RANDOM_STATE)
X_scaled_resampled, y_vals_resampled = bsmote.fit_resample(X_scaled, y_vals)

# Run PCA on the resampled data
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled_resampled)
var_exp = pca.explained_variance_ratio_ * 100

# UMAP / t-SNE runs on the resampled data
if UMAP_AVAILABLE:
    print("Running UMAP on oversampled data...")
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE,
                        n_neighbors=30, min_dist=0.1, n_jobs=-1) 
    X_embed = reducer.fit_transform(X_scaled_resampled)
    embed_label = "UMAP"
    dim_labels = ("UMAP Dim 1", "UMAP Dim 2")
else:
    print("Running t-SNE on oversampled data...")
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, 
                max_iter=1000, n_jobs=-1) 
    X_embed = tsne.fit_transform(X_scaled_resampled)
    embed_label = "t-SNE"
    dim_labels = ("t-SNE Dim 1", "t-SNE Dim 2")

# --- Side-by-side: PCA and UMAP/t-SNE ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, coords, title, xl, yl in [
    (axes[0], X_pca, "PCA 2D Projection (Borderline-SMOTE)",
     f"PC1 ({var_exp[0]:.1f}% var)", f"PC2 ({var_exp[1]:.1f}% var)"),
    (axes[1], X_embed, f"{embed_label} 2D Projection (Borderline-SMOTE)",
     dim_labels[0], dim_labels[1])
]:
    # Plot using the new resampled targets (y_vals_resampled)
    for label, color, name in [(0, "#4C72B0", "Good"), (1, "#C44E52", "Defaulted")]:
        mask = y_vals_resampled == label
        
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   c=color, alpha=0.4, s=10 if label == 0 else 20,
                   label=f"{name} (n={mask.sum():,})",
                   zorder=2 if label == 1 else 1,
                   rasterized=True) 
        
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(f"Diagnostic 3c — {title}", fontweight="bold")
    ax.legend(fontsize=9, markerscale=2)

plt.tight_layout()
plt.show()

In [ ]:
# --- 1. SET UP ACTUAL DATASET ---
SAMPLE_SIZE = 15000

if len(X) > SAMPLE_SIZE:
    from sklearn.model_selection import train_test_split
    X_sample, _, y_sample, _ = train_test_split(
        X, y, train_size=SAMPLE_SIZE, stratify=y, random_state=RANDOM_STATE
    )
else:
    X_sample, y_sample = X, y

if isinstance(X_sample, np.ndarray):
    X_sample = pd.DataFrame(X_sample)

# Fill missing values and scale
X_vals = X_sample.fillna(0).values
y_vals = y_sample.values if hasattr(y_sample, "values") else y_sample

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_vals)

# Split into Train and Test Sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_vals, test_size=0.25, stratify=y_vals, random_state=RANDOM_STATE
)


# --- 2. TRAIN BASELINE (BEFORE SMOTE) MODEL ---
clf_baseline = RandomForestClassifier(random_state=RANDOM_STATE)
clf_baseline.fit(X_train, y_train)

y_pred_base = clf_baseline.predict(X_test)
y_prob_base = clf_baseline.predict_proba(X_test)[:, 1]

# --- 3. TRAIN RESAMPLED (AFTER BORDERLINE-SMOTE) MODEL ---
bsmote = BorderlineSMOTE(kind='borderline-1', random_state=RANDOM_STATE)
X_train_res, y_train_res = bsmote.fit_resample(X_train, y_train)

clf_smote = RandomForestClassifier(random_state=RANDOM_STATE)
clf_smote.fit(X_train_res, y_train_res)

y_pred_smote = clf_smote.predict(X_test)
y_prob_smote = clf_smote.predict_proba(X_test)[:, 1]

# --- 4. CALCULATE CORE METRICS ---
metrics_list = ['Precision', 'Recall', 'F1-Score', 'G-Mean', 'ROC-AUC']

base_scores = [
    precision_score(y_test, y_pred_base, pos_label=1),
    recall_score(y_test, y_pred_base, pos_label=1),
    f1_score(y_test, y_pred_base, pos_label=1),
    geometric_mean_score(y_test, y_pred_base),
    roc_auc_score(y_test, y_prob_base)
]

smote_scores = [
    precision_score(y_test, y_pred_smote, pos_label=1),
    recall_score(y_test, y_pred_smote, pos_label=1),
    f1_score(y_test, y_pred_smote, pos_label=1),
    geometric_mean_score(y_test, y_pred_smote),
    roc_auc_score(y_test, y_prob_smote)
]

df_metrics = pd.DataFrame({
    'Metric': metrics_list * 2,
    'Score': base_scores + smote_scores,
    'Condition': ['Before SMOTE'] * 5 + ['After Borderline-SMOTE'] * 5
})

# --- 5. GENERATE SIDE-BY-SIDE BAR GRAPH ---
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    data=df_metrics, x='Metric', y='Score', hue='Condition',
    palette=['#4C72B0', '#C44E52']
)

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10, weight='bold')

plt.title('Model Performance Comparison: Before vs. After Borderline-SMOTE', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Score (0.0 to 1.0)', fontsize=12, fontweight='bold')
plt.xlabel('Evaluation Metrics (Minority Class Focused)', fontsize=12, fontweight='bold')
plt.ylim(0, 1.1)
plt.legend(title='Dataset State', loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

df_metrics

- **Precision (Baseline vs. After SMOTE):**  
  Before SMOTE, precision was **0.000** because the baseline model predicted **zero defaults** (it collapsed entirely under the class imbalance). This is the “zero‑prediction collapse” failure mode. After Borderline‑SMOTE, precision rose to **0.205**, meaning the model finally started flagging some defaults, though at the cost of more false positives.

- **Recall:**  
  Baseline recall was **0.000**, confirming the model caught **none** of the actual defaulters. After SMOTE, recall improved slightly to **0.026**, showing the model is at least identifying a small fraction of default cases. The treatment broke the collapse and allowed the model to recognize defaults.

- **F1‑Score:**  
  With the baseline at **0.000**, the F1‑Score after SMOTE (0.047) shows improvement. Even though the gain is small, it’s important because the model went from **no predictive ability** to at least balancing precision and recall.

- **G‑Mean:**  
  The baseline G‑Mean of **0.000** again reflects total failure. After SMOTE, G‑Mean rose to **0.162**, showing that the model now has some ability to balance accuracy across both classes.

- **ROC‑AUC:**  
  ROC‑AUC was **0.678** before SMOTE, which looks decent. But this is misleading because the model was simply predicting “all good borrowers.” After SMOTE, ROC‑AUC improved to **0.708**, and now the curve reflects a model that is actually distinguishing defaults from non‑defaults.


The most important finding in D4 is that the **baseline model completely failed on imbalanced data**. Precision, Recall, F1, and G‑Mean were all **exactly zero**, meaning the RandomForestClassifier predicted **zero defaults**. This collapse is the textbook reason why oversampling methods like Borderline‑SMOTE are necessary in credit risk modeling. (Elreedy & Atiya, 2019)

---

## Feature Engineering

These features are standard in credit risk modeling and supported by the EDA:
- **Credit burden:** `CREDIT_INCOME_RATIO = AMT_CREDIT/AMT_INCOME_TOTAL`
- **Repayment affordability:** `ANNUITY_INCOME_RATIO = AMT_ANNUITY/AMT_INCOME_TOTAL`
- **Loan duration in payment periods:** `CREDIT_TERM = AMT_CREDIT/AMT_ANNUITY`
- `AGE_YEARS = abs(DAYS_BIRTH)/365`
- `YEARS_EMPLOYED = abs(DAYS_EMPLOYED)/365`

Moreover, correlations an KS statistics will be checked to confirm these features add predictive signal.

In [ ]:
df_cleaned['CREDIT_INCOME_RATIO'] = df_cleaned['AMT_CREDIT'] / df_cleaned['AMT_INCOME_TOTAL']
df_cleaned['ANNUITY_INCOME_RATIO'] = df_cleaned['AMT_ANNUITY'] / df_cleaned['AMT_INCOME_TOTAL']
df_cleaned['CREDIT_TERM'] = df_cleaned['AMT_CREDIT'] / df_cleaned['AMT_ANNUITY']
df_cleaned['AGE_YEARS'] = abs(df_cleaned['DAYS_BIRTH']) / 365
df_cleaned['YEARS_EMPLOYED'] = abs(df_cleaned['DAYS_EMPLOYED']) / 365

df_cleaned[['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','CREDIT_TERM','AGE_YEARS','YEARS_EMPLOYED']].head()

### Check Correlation

In [ ]:
plt.figure(figsize=(8,6))
corr = df_cleaned[['TARGET','CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','CREDIT_TERM','AGE_YEARS','YEARS_EMPLOYED']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation of Engineered Features with Default (TARGET)", fontsize=14, fontweight='bold')
plt.show()

The engineered features show **very weak linear correlations** with TARGET (between -0.08 and 0.01). This suggests they add little predictive power. However, correlation only measures **linear association**. The KS statistic captures non‑linear separation between good and bad borrowers, which is more relevant in credit risk. (Thomas et al., 2017)

### KS Statistic Check

In [ ]:
from scipy.stats import ks_2samp

for feature in ['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','CREDIT_TERM','AGE_YEARS','YEARS_EMPLOYED']:
    good = df_cleaned.loc[df['TARGET']==0, feature].dropna()
    bad = df_cleaned.loc[df['TARGET']==1, feature].dropna()
    ks_stat, p_val = ks_2samp(good, bad)
    print(f"{feature}: KS={ks_stat:.3f}, p-value={p_val:.3e}")

- `AGE_YEARS` and `YEARS_EMPLOYED` have very weak separation.
- `CREDIT_TERM` has a weak but non-zero separation.
- `CREDIT_INCOME_RATIO` and `ANNUITY_INCOME_RATIO` have moderate separation.

While none of these features reach high separation, `AGE_YEARS` and `YEARS_EMPLOYED` show moderate discriminatory power that correlation would miss. Although the linear correlations are weak, the KS statistics justify retaining `CREDIT_TERM`, `AGE_YEARS`, and `YEARS_EMPLOYED` in the final pipeline. They provide non‑linear separation that supports credit risk modeling, and their WOE encoding ensures monotonic, interpretable risk trends.

---

## Weight of Evidence (WOE) & Information Value (IV)

In order to make the model more interpretable and credit-specific, binning the continous variables and encoding them using WOE is the industry-standard step. Moreover, the IV will also be calculated to measure the predictive power:
- Drop features with IV <0.02 (no predictive power)
    - 0.02-0.1 = weak
    - 0.1-0.3 = medium
    - 0.3-0.5 = strong
    - Greater than 0.5 = predictive
        - Note: Check for data leakage

### Bin Continuous Variables

In [ ]:
# Bin engineered features into 5 group
df_cleaned['CREDIT_INCOME_BIN'] = pd.qcut(df_cleaned['CREDIT_INCOME_RATIO'], q=5, duplicates='drop')
df_cleaned['ANNUITY_INCOME_BIN'] = pd.qcut(df_cleaned['ANNUITY_INCOME_RATIO'], q=5, duplicates='drop')
df_cleaned['CREDIT_TERM_BIN'] = pd.qcut(df_cleaned['CREDIT_TERM'], q=5, duplicates='drop')
df_cleaned['AGE_YEARS_BIN'] = pd.qcut(df_cleaned['AGE_YEARS'], q=5, duplicates='drop')
df_cleaned['YEARS_EMPLOYED_BIN'] = pd.qcut(df_cleaned['YEARS_EMPLOYED'], q=5, duplicates='drop')

### WOE & IV Functions

In [ ]:
def calc_woe_iv(df_cleaned, feature, target='TARGET'):
    lst = []
    for val in df_cleaned[feature].unique():
        good = df_cleaned[(df_cleaned[feature]==val) & (df_cleaned[target]==0)].shape[0]
        bad = df_cleaned[(df_cleaned[feature]==val) & (df_cleaned[target]==1)].shape[0]
        dist_good = good / df_cleaned[df_cleaned[target]==0].shape[0]
        dist_bad = bad / df_cleaned[df_cleaned[target]==1].shape[0]
        woe = np.log((dist_good + 1e-6) / (dist_bad + 1e-6))  # avoid div by zero
        iv = (dist_good - dist_bad) * woe
        lst.append([val, good, bad, dist_good, dist_bad, woe, iv])
    return pd.DataFrame(lst, columns=['Bin','Good','Bad','Dist_Good','Dist_Bad','WOE','IV'])

credit_income_woe_iv = calc_woe_iv(df_cleaned, 'CREDIT_INCOME_BIN')
annuity_income_woe_iv = calc_woe_iv(df_cleaned, 'ANNUITY_INCOME_BIN')
credit_term_woe_iv = calc_woe_iv(df_cleaned, 'CREDIT_TERM_BIN')
age_years_woe_iv = calc_woe_iv(df_cleaned, 'AGE_YEARS_BIN')
years_employed_woe_iv = calc_woe_iv(df_cleaned, 'YEARS_EMPLOYED_BIN')

print(f'CREDIT_INCOME:\n {credit_income_woe_iv}')
print(f'ANNUITY_INCOME:\n {annuity_income_woe_iv}')
print(f'CREDIT_TERM:\n {credit_term_woe_iv}')
print(f'AGE_YEARS:\n {age_woe_iv}')
print(f'YEARS_EMPLOYED:\n {years_woe_iv}')

print("\nTotal IV for CREDIT_INCOME:", credit_income_woe_iv['IV'].sum())
print("Total IV for ANNUITY_INCOME:", annuity_income_woe_iv['IV'].sum())
print("Total IV for CREDIT_TERM:", credit_term_woe_iv['IV'].sum())
print("Total IV for AGE_YEARS:", age_years_woe_iv['IV'].sum())
print("Total IV for YEARS_EMPLOYED:", years_employed_woe_iv['IV'].sum())

**WOE**
- `CREDIT_INCOME`, `ANNUITY_INCOME` and `CREDIT_TERM` have more bad borrowers than good, as evidence by having more negative bins than positive.
- `AGE_YEARS` and `YEARS_EMPLOYED` have more good borrowers.

**IV**
- `CREDIT_INCOME` and `ANNUITY_INCOME` have an IV <0.02, indicating that they have no predictive power. These features will be **dropped**.
- `CREDIT_TERM` and `AGE_YEARS` have an IV between 0.02-0.1, indicating that they are **weak predictors** . These features will be retained.
- `YEARS_EMPLOYED` has an IV between 0.1-0.3, indicating that it is a **medium predictor**. This feature will be retained.

### Drop Features with IV <0.02

In [ ]:
iv_scores = {}
for feature in ['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','CREDIT_TERM','AGE_YEARS','YEARS_EMPLOYED']:
    df_cleaned[feature+'_BIN'] = pd.qcut(df_cleaned[feature], q=5, duplicates='drop')
    iv_table = calc_woe_iv(df_cleaned, feature+'_BIN')
    iv_scores[feature] = iv_table['IV'].sum()

# Keep only features with IV >= 0.02
selected_features = [f for f, iv in iv_scores.items() if iv >= 0.02]
print("Selected features:", selected_features)

The scorecard becomes interpretable through WOE encoding. Additionaly, by filtering features based on their IV, the features that remain are only those with predictive power. These features will be used in the final model pipeline.

---

## Final Pipeline

A final model using a proper pipeline will be trained. The pipeline chains:
1. Imputation
2. WOE encoding
3. Scaling
4. Borderline-SMOTE
5. Logistic Regression (or XGBClassifier)

The result will be evaluated on a test set using ROC-AUC and the Gini coefficients.

### Setup the Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import BorderlineSMOTE

# --- Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    df_cleaned[selected_features], df_cleaned['TARGET'],
    test_size=0.25, stratify=df_cleaned['TARGET'], random_state=42
)

# --- Build Pipeline ---
pipeline = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),   # handle missing values
    ('scaler', StandardScaler()),                   # scale features
    ('smote', BorderlineSMOTE(kind='borderline-1', random_state=42)),  # resample
    ('clf', LogisticRegression(max_iter=1000, solver='lbfgs'))         # final model
])

# --- Train ---
pipeline.fit(X_train, y_train)

# --- Predict ---
y_prob = pipeline.predict_proba(X_test)[:, 1]

# --- Evaluate ---
roc_auc = roc_auc_score(y_test, y_prob)
gini = 2 * roc_auc - 1

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"Gini Coefficient: {gini:.3f}")

### Interpretation

**ROC-AUC** measures how well the model separates good vs. bad borrowers. A value of 0.5 indicates random guessing, 0.7-0.8 is decent, and 0.8+ is strong. `ROC-AUC = 0.584` shows weak discriminatory power. It is better than random guessing but significantly lower than the 0.708 ROC‑AUC observed in D4: Baseline Classifier Behaviour.

The drop happened because the D4 test used all 80 features from `df_cleaned`, which maximized raw predictive accuracy. The final pipeline, however, deliberately restricted itself to 3 selected features (`CREDIT_TERM`, `AGE_YEARS`, `YEARS_EMPLOYED`) that passed WOE/IV filtering. This reduction in feature space naturally lowers predictive accuracy because many moderately predictive variables (e.g., `EXT_SOURCE_2`, `EXT_SOURCE_3`) were excluded.

This drop reflects the trade-off between raw predictive accuracy and scorecard interpretability, and to align with credit risk convention:
- WOE encoding ensures monotonic, interpretable risk trends
- IV filtering removes weak predictors
- A smaller, justified feature set supports regulatory compliance

**Gini Coefficient** measures how far a distribution deviates from a perfectly equal distribution. A coefficient of 0 represents perfect inequality, and 1 indicates otherwise. Since `Gini Coefficient = 0.169`, this indicates **weak discriminatory power**: the model is better than random guessing but far below the 0.3–0.5 range typically expected in retail credit risk scorecards. (MetricGate, 2025)  

This result reflects the severe IR and overlap diagnosed earlier. Borderline-SMOTE improved the model by breaking the zero-prediction collapse, but stronger features and potentially more advanced classifiers are needed to raise the Gini into an acceptable range.  


The Gini confirms that while the pipeline is methodologically correct, it remains a proof-of-concept. Further feature engineering and model refinement are required for production-level performance.

---

## Conclusion

The pipeline ensures reproducibility and proper sequencing of preprocessing steps; Borderline-SMOTE improves minority class representation, Logistic Regression provides interpretable coefficients for the development of the scorecard, and the ROC-AUC and Gini confirms the discrepancy power of the final model.

---

## References

Elreedy, D., & Atiya, A. F. (2019). A Comprehensive Analysis of Synthetic Minority Oversampling Technique (SMOTE) for handling class imbalance. Information Sciences, 505, 32–64. https://doi.org/10.1016/j.ins.2019.07.070

MetricGate. (2025). MetricGate. https://metricgate.com/blogs/credit-scoring-logistic-regression/

Thomas, L. C., Crook, J. N., & Edelman, D. B. (2017). Credit scoring and its applications. Society For Industrial And Applied Mathematics.